In [52]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [53]:
import dataclasses
import os


@dataclasses.dataclass
class AppConfig:
    document_intelligence_endpoint: str
    document_intelligence_api_key: str
    input_pdf_file_path: str

    # DPI scale factor to 72 DPI
    dpi_scale: int


def load_config() -> AppConfig:
    endpoint = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT")
    api_key = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY")

    if not endpoint:
        raise ValueError("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT is not set")
    if not api_key:
        raise ValueError("AZURE_DOCUMENT_INTELLIGENCE_KEY is not set")

    return AppConfig(
        document_intelligence_endpoint=endpoint,
        document_intelligence_api_key=api_key,
        input_pdf_file_path="MelanconFeeleySerranoSLE23_page1.pdf",
        dpi_scale=10,
    )


config = load_config()


In [ ]:
from azure.ai.documentintelligence.aio import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import (
    AnalyzeResult,
    DocumentContentFormat,
)
from azure.core.credentials import AzureKeyCredential

document_intelligence_client = DocumentIntelligenceClient(
    endpoint=config.document_intelligence_endpoint,
    credential=AzureKeyCredential(config.document_intelligence_api_key),
)

os.mkdir("figures")
async with document_intelligence_client:
    with open(config.input_pdf_file_path, "rb") as f:
        poller = await document_intelligence_client.begin_analyze_document(
            model_id="prebuilt-layout",
            body=f,
            output_content_format=DocumentContentFormat.MARKDOWN,
        )
    result: AnalyzeResult = await poller.result()

FileExistsError: [WinError 183] 当文件已存在时，无法创建该文件。: 'figures'

In [ ]:
import pypdfium2 as pdfium

pdf = pdfium.PdfDocument(config.input_pdf_file_path, autoclose=True)
try:
    page = pdf.get_page(0)
    try:
        page_width, page_height = page.get_size()
        bitmap = page.render(
            scale=config.dpi_scale,
            may_draw_forms=True,
        )
        try:
            figure_pil = bitmap.to_pil()
        finally:
            bitmap.close()
    finally:
        page.close()
finally:
    pdf.close()

In [ ]:
assert result.figures
assert len(result.figures) == 1
assert result.figures[0].bounding_regions
assert len(result.figures[0].bounding_regions) == 1

polygon_inch = result.figures[0].bounding_regions[0].polygon

DPI_BASE = 72

dpi = config.dpi_scale * DPI_BASE
polygon_pixel = [
    (polygon_inch[i] * dpi, polygon_inch[i + 1] * dpi)
    for i in range(0, len(polygon_inch), 2)
]

In [ ]:
from PIL import ImageDraw

fig = figure_pil.copy()

xs, ys = zip(*polygon_pixel)
bbox = [min(xs), min(ys), max(xs), max(ys)]

draw = ImageDraw.Draw(fig)
draw.rectangle(bbox, outline="red", width=5)

fig.save("figures/page_0_figure_0.png")

In [58]:
from PIL import Image

mask = Image.new("L", figure_pil.size, 0)
ImageDraw.Draw(mask).polygon(polygon_pixel, outline=1, fill=255)

cropped = Image.new("RGBA", figure_pil.size)
cropped.paste(figure_pil, mask=mask)

final_img = cropped.crop(bbox)

final_img.save("figures/figure_0.png")

In [60]:
import re

pattern = re.compile(r"<figure>.*?</figure>", re.DOTALL)

In [ ]:
acc = []
last_end = 0

for idx, m in enumerate(pattern.finditer(result.content)):
    acc.append(result.content[last_end : m.start()])
    acc.append(f"![alt](figures/figure_{idx}.png)")
    last_end = m.end()

acc.append(result.content[last_end:])

text = "".join(acc)

In [66]:
from IPython.display import display_markdown

display_markdown(text, raw=True)

with open(config.input_pdf_file_path + ".md", "w") as f:
    f.write(text)

<!-- PageHeader="An Executable Semantics for Faster Development of Optimizing Python Compilers" -->
<!-- PageHeader="SLE '23, October 23-24, 2023, Cascais, Portugal" -->


![alt](figures/figure_0.png)


it does not allow a comparison with PyPy, which treats the
kernel of many of our benchmarks as dead code. Neither Zipi
nor CPython do this, so every operation is actually executed.
Figure 18 shows the results of our microbenchmarks. All
microbenchmarks are described in more details in [15].

Microbenchmarks indicate that behavior optimizations
provide a significant performance boost for binary operators
on small integers (between 15x and 30x faster) and floats
(between 3.0x and 7.2x), truthiness of bools (between 8.7x
and 14x), ints (20x) and strs (between 4.2x and 6.7x), and
comparison between ints (18x) and floats (7.2x).

Performance improvements from other optimizations
unrelated to behaviors also show up in the microbenchmarks.
For instance, assignment to global variables, function calls
and iteration on built-in types are all faster than with
CPython. On the other hand, some microbenchmarks
display poor performance. Those are unoptimized features
that we implemented in a naive way, such as function calls
with keyword arguments.


# 7.2 Benchmarks

We compared Zipi to CPython and PyPy using custom
benchmarks and benchmarks from PyPerformance, an
authoritative suite of benchmarks for Python [26]. Zipi
being at an early development stage, only four benchmarks
from PyPerformance are supported at the moment, hence
the need for custom benchmarks.

Our custom benchmarks include ack, fib, queens, bague
and sieve. The code for all custom benchmarks is available
in [15]. Benchmarks from PyPerformance include deltablue,
fannkuch, richards and float and are available online [30].
Each benchmark is executed once using parameters that
result in a run time on the order of one second on CPython.
Figure 19 compares the execution time of Zipi and PyPy
using the CPython execution time as a baseline.

Zipi fares especially well on programs that extensively use
small integer arithmetic: ack (38x faster than CPython), fib
(24x) and queens (14x) execute faster than with PyPy. The
bague (3.9x) and sieve (1.2x) benchmarks are slightly faster
than CPython with Zipi. These benchmarks use small integer
arithmetic, but also list and attribute access. The behavior
optimization has a noticeable but limited effect in those cases.
Finally, fannkuch (0.8x), richards (0.7x), deltablue (0.5x)
and float (0.3x) execute slower than with CPython. These
benchmarks make extensive use of user-defined types, which
we did not optimize, our focus being on built-in types.

Overall, Zipi's performance on benchmarks making
intensive use of small integer arithmetic rival with PyPy.
Yet, this speedup does not translate to benchmarks that
make a limited use of arithmetic. This is expected since
behaviors specifically target arithmetic. We wish to extend
the behavior optimization to other operations in the future
to further analyze its impact on performance.


## 7.3 Threats to Validity

The validity of our results faces the common potential issues
of assessing the performance of a prototype compiler.

Despite implementing Python's core features, including
those identified as the main source of overhead in CPython
(see Section 2), Zipi only supports a subset of the language.
It lacks features such as threads, async functions, and most
of the standard library. It remains to measure the impact of
introducing these features in our prototype.

Our benchmarks show a clear performance increase when
executing arithmetic-heavy programs. Nonetheless, the
absence of most modules from Python's standard library
limits our ability to measure the extent of this speed up on
real-life programs. The PyPerformance benchmark suite also
makes use of external libraries (such as django, a high-level
